<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 05 · Numerical Computing with NumPy

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the chapter examples in a Colab-ready format so that you
can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time so the objects or plots are
  created in order.
- Add your own cells for experiments or refactorings.
- Use the book text for the surrounding explanation and context.


Numerical computing is at the core of quantitative finance workflows: pricing,
risk, backtesting, and simulation all rely on fast operations over large
arrays of numbers. This chapter introduces `NumPy`, the array-based numerical
foundation that underpins later chapters on `pandas`, time series, simulation,
and machine learning.


# Why NumPy for Finance


Finance code uses `NumPy` because it turns batches of numbers into fast
vectorized operations.


# Creating Your First Arrays


Start with `ndarray` objects so you can treat lists of prices or returns as
numeric arrays.


In [ ]:
import numpy as np  # Import `NumPy` using the standard `np` alias.

In [ ]:
# Start from a plain Python list of prices.
prices_list = [100.0, 101.5, 103.2, 102.8]


In [ ]:
# Convert the list into an `ndarray`; this will be the default representation
# for numeric work.
prices = np.array(prices_list)

In [ ]:
prices

## Array Attributes and Dtypes


Shape, dtype, and dimensionality control how arrays behave in later
calculations.


In [ ]:
prices.dtype  # The `dtype` describes the type and precision of each element.

In [ ]:
# `ndim` is the number of dimensions (axes); here a one-dimensional vector.
prices.ndim

In [ ]:
# `shape` is a tuple with one entry per dimension; `(4,)` means four elements
# in one axis.
prices.shape

In [ ]:
prices.size  # `size` is the total number of elements in the array.

In [ ]:
# `itemsize` is the number of bytes per element; this array has 4 elements × 8
# bytes each = 32 bytes in total.
prices.itemsize

In [ ]:
# Explicitly request 64-bit integers for integer quantities such as share
# counts.
qty = np.array([100, 250, 75], dtype=np.int64)

In [ ]:
qty.dtype

## Array Creation Routines


`zeros`, `ones`, `arange`, `linspace`, and friends generate test data and
structured numeric inputs.


In [ ]:
# `np.zeros` creates a one-dimensional array of zeros; pass an integer for the
# length.
zeros = np.zeros(5)

In [ ]:
# Passing a tuple creates a 2D array with the given shape.
ones = np.ones((2, 3))


In [ ]:
# `np.eye` creates an identity matrix, useful in linear algebra.
eye = np.eye(3)


In [ ]:
# `np.linspace` creates a grid of evenly spaced values between start and end
# (inclusive).
grid = np.linspace(0.0, 1.0, num=5)

In [ ]:
zeros

In [ ]:
ones

In [ ]:
eye

In [ ]:
grid

# Working with Dimensions and Indexing


Array rank and indexing determine how you extract scalars, rows, and slices.


## 1D and 2D Indexing


One- and two-dimensional indexing are the base operations for selecting values
from arrays.


In [ ]:
# A simple 2D array: two rows (e.g., assets) and three columns (e.g., time
# points).
prices_2d = np.array([
    [100.0, 101.5, 103.2],
    [ 99.5, 100.8, 102.0],
])

In [ ]:
# Shape `(2, 3)` means axis 0 has length 2 and axis 1 has length 3.
prices_2d.shape


In [ ]:
prices_2d[0]  # Single index selects a row; result is a 1D view.

In [ ]:
prices_2d[1, 2]  # Two indices select a single element (row 1, column 2).

In [ ]:
# The colon (`:`) selects "all rows"; this expression extracts the second
# column.
prices_2d[:, 1]

## Slicing and Views


Slices usually share memory with the original array, so updates can propagate
unexpectedly.


In [ ]:
# Original price vector with four elements.
spot = np.array([100.0, 101.0, 102.0, 103.0])


In [ ]:
window = spot[1:3]  # Slice selecting indices 1 and 2 (end index is exclusive).

In [ ]:
window

In [ ]:
# Changing the slice modifies the original `spot` array, because `window` is a
# view.
window[0] = 999.0

In [ ]:
spot

In [ ]:
spot = np.array([100.0, 101.0, 102.0, 103.0])

In [ ]:
# `.copy()` forces allocation of a new array; changes no longer affect the
# original.
window_copy = spot[1:3].copy()

In [ ]:
window_copy[0] = 999.0

In [ ]:
spot

## Aggregations and the Axis Argument


Reduction functions depend on axis selection, which changes whether you
collapse rows or columns.


In [ ]:
returns = np.array([  # Two assets over three days of returns.
    [0.01,  0.02, -0.005],
    [-0.01, 0.015,  0.0  ],
])

In [ ]:
returns.mean()  # Mean over all elements in the array.

In [ ]:
# Mean per column (axis 0): aggregate across assets for each day.
returns.mean(axis=0)


In [ ]:
# Mean per row (axis 1): aggregate across days for each asset.
returns.mean(axis=1)


# Boolean Masks and Conditional Logic


Boolean arrays let you filter and transform data without explicit Python
loops.


## Building Boolean Masks


Mask expressions turn conditions like thresholds or missing values into array-
wide filters.


In [ ]:
prices = np.array([100.0, 101.5, 99.0, 102.0])  # A simple vector of prices.

In [ ]:
mask = prices > 100.0  # Element-wise comparison returns a Boolean mask.

In [ ]:
mask

In [ ]:
# Using the mask as an index selects only elements where the condition is
# `True`.
prices[mask]

In [ ]:
cheap = prices < 101.0

In [ ]:
# Combine two conditions with element-wise logical `and`.
mid_range = (prices >= 100.0) & (prices <= 102.0)

In [ ]:
cheap

In [ ]:
mid_range

## np.where for Vectorized If-Else


`np.where` chooses values elementwise and keeps the logic in array form.


In [ ]:
prices = np.array([100.0, 101.5, 99.0, 102.0])

In [ ]:
threshold = 100.0

In [ ]:
# Create a simple trading signal: `1` if price is above the threshold, `-1`
# otherwise.
signal = np.where(prices > threshold, 1, -1)

In [ ]:
signal

In [ ]:
# Vector of base returns before any adjustment.
base = np.array([0.01, -0.02, 0.015, -0.005])


In [ ]:
# Apply a simple haircut with `np.where`: reduce positive values by 10% and
# increase negative ones by 10%.
adjusted = np.where(base >= 0.0, base * 0.9, base * 1.1)

In [ ]:
adjusted

# Vectorization and Broadcasting


Vectorization and broadcasting remove Python-level loops from common finance
calculations.


## Replacing Loops with Array Operations


Array operations are usually shorter and faster than manual accumulation
loops.


In [ ]:
prices = np.array([100.0, 101.5, 103.0, 102.0])

In [ ]:
returns_loop = []  # Initialize a Python list to collect results.

In [ ]:
for i in range(1, len(prices)):
    # Explicit loop computes the return step by step.
    r = (prices[i] / prices[i - 1]) - 1
    returns_loop.append(r)

In [ ]:
returns_loop

In [ ]:
# Slice the price array into a "future" part and a "past" part and divide
# element-wise.
returns_vec = prices[1:] / prices[:-1] - 1

In [ ]:
returns_vec

## Broadcasting Rules by Example


Broadcasting works when dimensions line up or can be expanded safely.


In [ ]:
x = np.array([  # A 2×3 matrix of observations.
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])

In [ ]:
col_means = x.mean(axis=0)  # Column means have shape `(3,)`.

In [ ]:
# Subtracting a `(3,)` array from a `(2, 3)` array broadcasts the vector
# across rows.
x_centered = x - col_means

In [ ]:
col_means

In [ ]:
x_centered

In [ ]:
daily_returns = np.array([0.01, -0.005, 0.002])  # Per-asset returns over a day.

In [ ]:
# Column vector of exposures for two portfolios.
exposure = np.array([[1.0], [0.5]])


In [ ]:
# Broadcasting multiplies each row of `exposure` across the return vector.
portfolio_returns = exposure * daily_returns

In [ ]:
portfolio_returns

# Random Numbers and Simple Simulations


Random draws let you model uncertainty, price paths, and sampling error.


## Using the Modern Generator API


The `Generator` API gives you reproducible random streams with explicit state.


In [ ]:
# Create a `Generator` with a fixed seed for reproducible results.
rng = np.random.default_rng(seed=42)

In [ ]:
# Draw standard normal samples using the generator.
normals = rng.standard_normal(size=5)


In [ ]:
normals

## A One-Step Monte Carlo Example


A single-step simulation shows how random shocks feed into a price model.


In [ ]:
S0 = 100.0  # Initial price.

In [ ]:
mu = 0.05  # Drift parameter (annualized expected return).

In [ ]:
sigma = 0.2  # Volatility parameter (annualized standard deviation).

In [ ]:
T = 1.0  # Time horizon in years.

In [ ]:
rng = np.random.default_rng(seed=123)

In [ ]:
z = rng.standard_normal(size=10_000)  # Simulated standard normal shocks.

In [ ]:
# Vectorized GBM formula applied to all shocks at once.
ST = S0 * np.exp((mu - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z)

In [ ]:
ST.mean(), ST.std()

In [ ]:
# Analytical expectation of the terminal price under this GBM
# parameterization.
expected_mean = S0 * np.exp(mu * T)

In [ ]:
# Check that the simulated mean is within a small relative tolerance of the
# analytical value; if the assert fails, either the implementation or the
# parameters need a second look.
assert np.isclose(ST.mean(), expected_mean, rtol=0.02)

# Performance and Memory Layout


Array contiguity and dtype choices affect both speed and memory use.


## Comparing Python Loops and NumPy


A direct timing comparison shows why vectorized operations matter.


In [ ]:
n = 1_000_000

In [ ]:
data = list(range(n))  # Build a plain Python list of integers.

In [ ]:
# Build a `NumPy` array of doubles with the same logical values.
arr = np.arange(n, dtype=np.float64)


In [ ]:
# Time a loop-based list comprehension that scales each element; on the
# reference machine this took about 0.020 s per run for one million elements
# (≈20 ms).
get_ipython().run_line_magic("timeit", "[x * 0.01 for x in data]")


In [ ]:
# Time the equivalent vectorized array operation, which delegates the work to
# optimized C code; in the same environment this took about 0.00013 s per run
# (≈0.13 ms).
get_ipython().run_line_magic("timeit", "arr * 0.01")


## Row-Major and Column-Major Access


Access order matters because it changes how efficiently the CPU walks through
memory.


In [ ]:
# Choose a moderately large matrix shape that resembles, for example, "many
# dates by many assets."
m, n = 2_000, 1_000

In [ ]:
rng = np.random.default_rng(seed=42)

In [ ]:
# Draw standard normal samples into a default C-ordered array (row-major).
c_order = rng.standard_normal(size=(m, n))

In [ ]:
# Create a column-major view of the same data, suitable for Fortran-style
# libraries.
f_order = np.asfortranarray(c_order)

In [ ]:
# Sum across rows (axis 0) for the C-ordered array; in our measurements this
# was on the order of a few tenths of a millisecond.
get_ipython().run_line_magic("timeit", "c_order.sum(axis=0)")


In [ ]:
# Sum across columns (axis 1) for the C-ordered array; the timings were of
# similar magnitude but can differ slightly due to access patterns.
get_ipython().run_line_magic("timeit", "c_order.sum(axis=1)")


In [ ]:
# Sum across rows for the F-ordered array, where column-wise access is more
# cache-friendly.
get_ipython().run_line_magic("timeit", "f_order.sum(axis=0)")


In [ ]:
# Sum across columns for the F-ordered array; depending on the shape and
# platform, this can be faster than the corresponding C-ordered operation.
get_ipython().run_line_magic("timeit", "f_order.sum(axis=1)")


# Structured Arrays for Tabular Data


Structured arrays store named columns inside a single `ndarray`.


## Defining a Structured dtype


A structured dtype describes each field's name and type.


In [ ]:
# Define a structured dtype with three fields: `symbol`, `price`, and
# `volume`.
dtype = np.dtype([("symbol", "U6"), ("price", "f8"), ("volume", "i8")])

In [ ]:
quotes = np.array(
    [("AAPL", 180.0, 1_000), ("MSFT", 350.0, 500), ("GOOG", 140.0, 2_000)],
    dtype=dtype,
)  # Build an array of records using that dtype; each tuple becomes one row.

In [ ]:
# Field indexing returns a regular `ndarray` view of the `price` column.
quotes["price"]


In [ ]:
# The same mechanism gives you vectorized access to the `symbol` field.
quotes["symbol"]


## Filtering with Structured Arrays


Field-based filters let you work with record-like arrays without converting to
a table.


In [ ]:
# Use a Boolean mask on the `volume` field to select large trades only.
big = quotes[quotes["volume"] >= 1_000]

In [ ]:
# Select the `symbol` field from the filtered array to see which instruments
# passed the threshold.
big["symbol"]

# Interoperability with Python and pandas


You can move between native Python containers, `ndarray` objects, and `pandas`
structures when needed.


## Converting Between Lists and Arrays


Conversion is the bridge between Python-native data and vectorized numerical
code.


In [ ]:
prices_list = [100.0, 101.5, 103.0]

In [ ]:
prices_arr = np.array(prices_list)  # Build an `ndarray` from a Python list.

In [ ]:
# Convert an `ndarray` back to a plain list when needed (for example, for JSON
# serialization).
prices_back = prices_arr.tolist()

In [ ]:
prices_arr

In [ ]:
prices_back

In [ ]:
import numpy.typing as npt

In [ ]:
def scale_returns(
    rets: npt.NDArray[np.floating], factor: float
# Annotate both the `rets` argument and the return type as floating-point
# `ndarray` objects, which helps static analyzers and editors catch accidental
# misuse.
) -> npt.NDArray[np.floating]:
    """Scale a vector of returns by a constant factor."""
    return factor * rets

In [ ]:
scale_returns(np.array([0.01, -0.02]), 0.5)

## Bridging to pandas


`pandas` takes `NumPy` arrays and adds labels, indexes, and table semantics.


In [ ]:
import pandas as pd  # Import `pandas` with its conventional alias.

In [ ]:
prices = np.array([100.0, 101.5, 103.0, 102.0])

In [ ]:
dates = pd.date_range("2026-01-01", periods=4, freq="B")

In [ ]:
# Build a time-indexed `DataFrame` from a `NumPy` array.
df = pd.DataFrame({"price": prices}, index=dates)

In [ ]:
df

In [ ]:
# Extract an `ndarray` view of the underlying numeric data.
df["price"].to_numpy()


# Where We Are Heading Next


Chapter 6 extends these arrays into labeled tables and time-indexed market
data.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
